# Hourly rainfall prediction – Benue State (23 LGAs, 2020–2024)
**Target:** `rainfall_mm` (hourly). Full ML workflow: data audit → EDA → leakage-safe feature engineering → temporal split → baselines → Ridge / Random Forest / HistGradientBoosting / hurdle model → tuning → evaluation (regression + rain-occurrence classification) → feature importance → saving the champion models.

**Two scenarios**
- **A – weather only:** time, location, concurrent + lagged temperature / humidity / wind / dew-point. No rainfall history.
- **B – with rainfall history:** A + past rainfall (lags, rolling sums, hours since last rain, regional rain).

**Leakage decisions:** `rain_mm` is an exact duplicate of the target; `daily_rainfall_sum_mm` contains the target; `daily_temp_max_c`/`daily_temp_min_c` use the whole day (future hours). All four are excluded.

> Runtime: roughly 10–15 minutes on one CPU core (Random Forest and gradient-boosting fits dominate). Set `DATA` in the first code cell to your CSV path.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Helpers: loading, feature engineering, metrics

In [ ]:
"""Shared helpers: data loading, leakage-safe feature engineering, metrics."""
import os, numpy as np, pandas as pd

DATA = "benue_weather_with_time_and_rainfall.csv"   # <- path to your CSV
OUT = "rainfall_ml_outputs"
TARGET = "rainfall_mm"
TRAIN_END, VAL_YEAR, TEST_YEAR = 2022, 2023, 2024
MET = ["temperature_c", "humidity_percent", "wind_speed_kmh", "dewpoint_depr"]


def load_raw():
    df = pd.read_csv(DATA, parse_dates=["datetime"])
    return df.sort_values(["lga", "datetime"]).reset_index(drop=True)


def _lag_roll(df, col, lags=(), rolls=(), prefix=None):
    """Past-only lags / rolling statistics per LGA (rows are strictly hourly)."""
    prefix = prefix or col
    g = df.groupby("lga")[col]
    out = {}
    for l in lags:
        out[f"{prefix}_lag{l}"] = g.shift(l)
    if rolls:
        cs = g.cumsum()
        gcs = cs.groupby(df["lga"])
        for w in rolls:
            out[f"{prefix}_roll{w}"] = (gcs.shift(1) - gcs.shift(1 + w))  # sum of t-w .. t-1
    return pd.DataFrame(out)


def build_features(df):
    df = df.copy()
    # --- physically-motivated variable: dew-point depression (Magnus formula)
    a, b = 17.62, 243.12
    gam = np.log(df.humidity_percent.clip(1, 100) / 100) + a * df.temperature_c / (b + df.temperature_c)
    df["dewpoint_c"] = b * gam / (a - gam)
    df["dewpoint_depr"] = df.temperature_c - df.dewpoint_c

    feats = {}
    # --- calendar (cyclic); `year` deliberately excluded (cannot extrapolate)
    doy = df.datetime.dt.dayofyear
    feats["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
    feats["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
    feats["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    feats["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    base = df[["lat", "lon", "temperature_c", "humidity_percent", "wind_speed_kmh", "dewpoint_c", "dewpoint_depr"]]

    # --- regional (all-LGA) mean of concurrent weather
    reg = {}
    for c in ["temperature_c", "humidity_percent", "wind_speed_kmh"]:
        reg[f"reg_{c}"] = df.groupby("datetime")[c].transform("mean")

    # --- lagged / rolling meteorology (past only)
    parts = [base, pd.DataFrame(feats), pd.DataFrame(reg)]
    for c in MET:
        lr = _lag_roll(df, c, lags=(1, 3, 6, 24), rolls=(6, 24))
        for w in (6, 24):
            lr[f"{c}_roll{w}"] = lr[f"{c}_roll{w}"] / w   # -> rolling mean
        parts.append(lr)
    tmp = pd.concat(parts, axis=1)
    for c, l in [("temperature_c", 1), ("temperature_c", 3), ("humidity_percent", 1),
                 ("humidity_percent", 3), ("wind_speed_kmh", 1)]:
        tmp[f"{c}_d{l}"] = df[c] - tmp[f"{c}_lag{l}"]
    met_cols = list(tmp.columns)

    # --- rainfall memory (past only) -> scenario B
    rain = _lag_roll(df, TARGET, lags=(1, 2, 3, 6, 12, 24), rolls=(3, 6, 24, 72), prefix="rain")
    wet = (df[TARGET] > 0).astype(float)
    wet_prev = wet.groupby(df.lga).shift(1)
    idx = df.groupby("lga").cumcount().astype(float)
    last_wet = idx.where(wet_prev == 1).groupby(df.lga).ffill()
    rain["hours_since_rain"] = (idx - last_wet).fillna(720).clip(upper=720)
    regr = df.groupby("datetime")[TARGET].mean().sort_index()
    rain["reg_rain_lag1"] = df.datetime.map(regr.shift(1))
    rain["reg_rain_roll6"] = df.datetime.map(regr.shift(1).rolling(6).sum())
    rain_cols = list(rain.columns)

    X = pd.concat([tmp, rain], axis=1)
    meta = df[["datetime", "lga", "year", "month", "date", TARGET]].copy()
    # drop warm-up rows with incomplete lag windows (first 73 h per LGA)
    keep = (idx >= 73).values
    X, meta = X[keep].reset_index(drop=True), meta[keep].reset_index(drop=True)
    return X.astype("float32"), meta, met_cols, rain_cols


def split_masks(meta):
    tr = (meta.year <= TRAIN_END).values
    va = (meta.year == VAL_YEAR).values
    te = (meta.year == TEST_YEAR).values
    return tr, va, te


def reg_metrics(y, p, meta=None):
    y, p = np.asarray(y, float), np.asarray(p, float)
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    mae = float(np.mean(np.abs(y - p)))
    r2 = float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())
    out = {"RMSE": rmse, "MAE": mae, "R2": r2}
    if meta is not None:  # daily totals per LGA (hydrologically more meaningful)
        d = pd.DataFrame({"k": meta["lga"].values + meta["date"].astype(str).values, "y": y, "p": p}).groupby("k").sum()
        out["Daily_RMSE"] = float(np.sqrt(np.mean((d.y - d.p) ** 2)))
        out["Daily_R2"] = float(1 - ((d.y - d.p) ** 2).sum() / ((d.y - d.y.mean()) ** 2).sum())
    return out

os.makedirs(f"{OUT}/figs", exist_ok=True); os.makedirs(f"{OUT}/results", exist_ok=True); os.makedirs(f"{OUT}/models", exist_ok=True)

## 2. Load data and audit

In [ ]:
raw = load_raw()
print(raw.shape, raw.datetime.min(), "->", raw.datetime.max())
print("missing values:", int(raw.isna().sum().sum()), "| duplicate LGA-hours:", int(raw.duplicated(["lga","datetime"]).sum()))
print("rain_mm identical to rainfall_mm:", bool((raw.rainfall_mm == raw.rain_mm).all()))
print("share of zero-rain hours: %.1f%%" % (100*(raw.rainfall_mm == 0).mean()))
raw.describe().T

## 3. Exploratory data analysis

In [ ]:
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
df = raw; F = f"{OUT}/figs"
y = df[TARGET]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].hist(y[y > 0], bins=60, color="#2a6f97"); ax[0].set_yscale("log"); ax[0].set_title(f"Non-zero hourly rainfall (zeros = {100*(y==0).mean():.1f}% of rows)"); ax[0].set_xlabel("mm/h")
m = df.groupby("month")[TARGET].mean(); ax[1].bar(m.index, m.values, color="#2a6f97"); ax[1].set_title("Mean hourly rainfall by month"); ax[1].set_xlabel("month")
h = df.groupby("hour")[TARGET].mean(); ax[2].plot(h.index, h.values, marker="o", color="#bc4749"); ax[2].set_title("Diurnal cycle (mean mm/h by hour)"); ax[2].set_xlabel("hour")
plt.tight_layout(); plt.savefig(f"{F}/01_target_distribution_seasonality.png"); plt.show()

num = ["rainfall_mm", "temperature_c", "humidity_percent", "wind_speed_kmh", "daily_rainfall_sum_mm", "daily_temp_max_c", "daily_temp_min_c"]
c = df[num].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.2, 5)); im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=45, ha="right"); ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
for i in range(len(num)):
    for j in range(len(num)): ax.text(j, i, f"{c.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, shrink=.8); ax.set_title("Spearman correlation (raw columns)"); plt.tight_layout(); plt.savefig(f"{F}/02_correlation.png"); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
a = df.groupby(["year"])[TARGET].sum() / df.lga.nunique(); ax[0].bar(a.index, a.values, color="#2a6f97"); ax[0].set_title("Mean annual rainfall per LGA (mm)")
l = df.groupby("lga")[TARGET].sum().div(5).sort_values(); ax[1].barh(l.index, l.values, color="#6a994e"); ax[1].set_title("Mean annual rainfall by LGA (mm)"); ax[1].tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.savefig(f"{F}/03_annual_and_lga.png"); plt.show()
# audit
pd.Series({"rows": len(df), "lgas": df.lga.nunique(), "period": f"{df.datetime.min()} to {df.datetime.max()}",
           "missing_values": int(df.isna().sum().sum()), "duplicate_lga_hours": int(df.duplicated(['lga','datetime']).sum()),
           "rain_mm identical to rainfall_mm": bool((df.rainfall_mm == df.rain_mm).all()),
           "share_zero_hours_%": round(100*(y==0).mean(),1)}).to_csv(f"{OUT}/results/data_audit.csv")

## 4. Feature engineering (past-only lags, no leakage) and temporal split
Train 2020–2022 · Validation 2023 (tuning / early stopping / model selection) · Test 2024 (touched once).

In [ ]:
X, meta, met_cols, rain_cols = build_features(raw)
tr, va, te = split_masks(meta)
y = meta[TARGET].values.astype("float64")
print(X.shape, "| weather features:", len(met_cols), "| rain-history features:", len(rain_cols))
print("rows  train/val/test:", tr.sum(), va.sum(), te.sum())

## 5. Modelling
For each scenario: baselines (mean, climatology, persistence) → Ridge → Random Forest (120k-row subsample) → random hyper-parameter search for HistGradientBoosting on 2023 → HistGB with squared-error and Poisson loss (iteration count chosen on 2023) → **hurdle model** `P(rain>0) × E[mm | rain]` → rain/no-rain classification report (threshold maximising F1 on validation) → champion (lowest validation RMSE) refit on 2020–2023 and saved → permutation importance.

In [ ]:
import time, warnings, joblib
from sklearn.ensemble import HistGradientBoostingRegressor as HGBR, HistGradientBoostingClassifier as HGBC, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve, f1_score
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
T0 = time.time()
def log(*a): print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)

tr, va, te = split_masks(meta)
y = meta[TARGET].values.astype("float64")
rng = np.random.RandomState(42)
SCEN = {"A_weather_only": met_cols, "B_with_rain_history": met_cols + rain_cols}
results, preds_store, cls_results, importances, tuning_log = [], {}, [], {}, []

def best_stage(model, Xv, yv):
    errs = [np.sqrt(np.mean((yv - p) ** 2)) for p in model.staged_predict(Xv)]
    return int(np.argmin(errs)) + 1, float(np.min(errs))

for scen, cols in SCEN.items():
    log("=== scenario", scen, len(cols), "features")
    Xtr, Xva, Xte = X.loc[tr, cols], X.loc[va, cols], X.loc[te, cols]
    ytr, yva, yte = y[tr], y[va], y[te]
    mte, mva = meta[te], meta[va]
    P = {}

    def record(name, p_va, p_te, secs):
        m = reg_metrics(yte, p_te, mte); mv = reg_metrics(yva, p_va)
        results.append({"scenario": scen, "model": name, **{f"test_{k}": v for k, v in m.items()},
                        "val_RMSE": mv["RMSE"], "val_R2": mv["R2"], "fit_seconds": round(secs, 1)})
        P[name] = p_te
        log(f"{name:28s} test RMSE={m['RMSE']:.4f} MAE={m['MAE']:.4f} R2={m['R2']:.4f} dailyR2={m['Daily_R2']:.4f}")

    # ---------- baselines
    record("Baseline: train mean", np.full(va.sum(), ytr.mean()), np.full(te.sum(), ytr.mean()), 0)
    clim = pd.DataFrame({"k": meta.lga[tr].values + "_" + meta.month[tr].astype(str).values + "_" + meta.datetime.dt.hour[tr].astype(str).values, "y": ytr}).groupby("k").y.mean()
    key = lambda m_: (m_.lga.values + "_" + m_.month.astype(str).values + "_" + m_.datetime.dt.hour.astype(str).values)
    record("Baseline: climatology", pd.Series(key(mva)).map(clim).fillna(ytr.mean()).values,
           pd.Series(key(mte)).map(clim).fillna(ytr.mean()).values, 0)
    if scen.startswith("B"):
        record("Baseline: persistence", X.loc[va, "rain_lag1"].values, X.loc[te, "rain_lag1"].values, 0)

    # ---------- Ridge
    t = time.time()
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(Xtr, ytr)
    record("Ridge regression", np.clip(ridge.predict(Xva), 0, None), np.clip(ridge.predict(Xte), 0, None), time.time() - t)

    # ---------- Random forest (subsample for 1-core budget)
    t = time.time()
    sub = rng.choice(len(Xtr), 120_000, replace=False)
    rf = RandomForestRegressor(n_estimators=60, max_depth=16, min_samples_leaf=40, max_features=0.4, n_jobs=1, random_state=42)
    rf.fit(Xtr.iloc[sub], ytr[sub])
    record("Random forest", rf.predict(Xva), rf.predict(Xte), time.time() - t)

    # ---------- Hyper-parameter search (random search, temporal validation = 2023)
    t = time.time()
    sub = rng.choice(len(Xtr), 150_000, replace=False)
    grid = {"learning_rate": [0.05, 0.1, 0.15], "max_leaf_nodes": [15, 31, 63, 127], "min_samples_leaf": [50, 200, 500],
            "l2_regularization": [0.0, 1.0, 10.0], "max_features": [0.5, 0.8, 1.0]}
    best = None
    for i, prm in enumerate(ParameterSampler(grid, 8, random_state=7)):
        m = HGBR(loss="squared_error", max_iter=250, early_stopping=False, random_state=0, **prm).fit(Xtr.iloc[sub], ytr[sub])
        n, e = best_stage(m, Xva, yva)
        tuning_log.append({"scenario": scen, **prm, "best_iter": n, "val_RMSE": e})
        log(f"  tune {i}: {prm} -> val RMSE {e:.4f} @ {n}")
        if best is None or e < best[0]: best = (e, prm)
    prm = best[1]; log("best params", prm, f"(search {time.time()-t:.0f}s)")

    # ---------- Gradient boosting: squared error & Poisson
    fitted = {}
    for name, loss in [("HistGB (squared error)", "squared_error"), ("HistGB (Poisson)", "poisson")]:
        t = time.time()
        m = HGBR(loss=loss, max_iter=500, early_stopping=False, random_state=0, **prm).fit(Xtr, ytr)
        n, e = best_stage(m, Xva, yva)
        m = HGBR(loss=loss, max_iter=n, early_stopping=False, random_state=0, **prm).fit(Xtr, ytr)
        fitted[name] = (m, n)
        record(name, m.predict(Xva), m.predict(Xte), time.time() - t)

    # ---------- Hurdle model: P(rain) x E[amount | rain]
    t = time.time()
    clf = HGBC(max_iter=400, early_stopping=False, random_state=0, **prm).fit(Xtr, (ytr > 0).astype(int))
    stg = [brier_score_loss((yva > 0).astype(int), p[:, 1]) for p in clf.staged_predict_proba(Xva)]
    nc = int(np.argmin(stg)) + 1
    clf = HGBC(max_iter=nc, early_stopping=False, random_state=0, **prm).fit(Xtr, (ytr > 0).astype(int))
    wr = ytr > 0
    amt = HGBR(loss="gamma", max_iter=300, early_stopping=False, random_state=0, **{**prm, "min_samples_leaf": min(prm["min_samples_leaf"], 100)}).fit(Xtr[wr], ytr[wr])
    na, _ = best_stage(amt, Xva[yva > 0], yva[yva > 0])
    amt = HGBR(loss="gamma", max_iter=na, early_stopping=False, random_state=0, **{**prm, "min_samples_leaf": min(prm["min_samples_leaf"], 100)}).fit(Xtr[wr], ytr[wr])
    pva, pte = clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1]
    record("Hurdle (clf x amount)", pva * amt.predict(Xva), pte * amt.predict(Xte), time.time() - t)

    # ---------- Rain / no-rain classification report (from hurdle classifier)
    pr, rc, th = precision_recall_curve((yva > 0).astype(int), pva)
    f1s = 2 * pr * rc / (pr + rc + 1e-12); thr = float(th[np.argmax(f1s[:-1])])
    yb = (yte > 0).astype(int); pb = (pte >= thr).astype(int)
    tp, fp = int(((pb == 1) & (yb == 1)).sum()), int(((pb == 1) & (yb == 0)).sum())
    fn, tn = int(((pb == 0) & (yb == 1)).sum()), int(((pb == 0) & (yb == 0)).sum())
    cls_results.append({"scenario": scen, "threshold(val F1-opt)": thr, "ROC_AUC": roc_auc_score(yb, pte),
                        "PR_AUC": average_precision_score(yb, pte), "Brier": brier_score_loss(yb, pte),
                        "Brier_climatology": brier_score_loss(yb, np.full_like(pte, (ytr > 0).mean())),
                        "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1),
                        "F1": f1_score(yb, pb), "Accuracy": (tp + tn) / len(yb), "TP": tp, "FP": fp, "FN": fn, "TN": tn})
    log("classification:", {k: round(v, 4) if isinstance(v, float) else v for k, v in cls_results[-1].items()})
    P["_prob"] = pte

    # ---------- Champion: lowest validation RMSE among ML models -> refit on train+val
    ml = [r for r in results if r["scenario"] == scen and not r["model"].startswith("Baseline")]
    champ = min(ml, key=lambda r: r["val_RMSE"])["model"]
    log("champion by validation RMSE:", champ)
    trva = tr | va
    if champ.startswith("HistGB"):
        m0, n = fitted[champ]
        loss = "poisson" if "Poisson" in champ else "squared_error"
        final = HGBR(loss=loss, max_iter=int(n * 1.15), early_stopping=False, random_state=0, **prm).fit(X.loc[trva, cols], y[trva])
        pf = final.predict(Xte)
        log("champion refit on 2020-2023, test:", {k: round(v, 4) for k, v in reg_metrics(yte, pf, mte).items()})
        results.append({"scenario": scen, "model": f"FINAL {champ} (refit 2020-23)", **{f"test_{k}": v for k, v in reg_metrics(yte, pf, mte).items()}, "val_RMSE": np.nan, "val_R2": np.nan, "fit_seconds": np.nan})
        P["FINAL"] = pf
        joblib.dump({"model": final, "features": cols, "params": prm, "n_iter": int(n * 1.15), "loss": loss},
                    f"{OUT}/models/rainfall_{scen}.joblib", compress=3)
        imp_model = final
    elif champ.startswith("Hurdle"):
        ap = {**prm, "min_samples_leaf": min(prm["min_samples_leaf"], 100)}
        clf_f = HGBC(max_iter=int(nc * 1.15), early_stopping=False, random_state=0, **prm).fit(X.loc[trva, cols], (y[trva] > 0).astype(int))
        wr2 = trva & (y > 0)
        amt_f = HGBR(loss="gamma", max_iter=int(na * 1.15), early_stopping=False, random_state=0, **ap).fit(X.loc[wr2, cols], y[wr2])
        pf = clf_f.predict_proba(Xte)[:, 1] * amt_f.predict(Xte)
        mf = reg_metrics(yte, pf, mte)
        log("champion refit on 2020-2023, test:", {k: round(v, 4) for k, v in mf.items()})
        results.append({"scenario": scen, "model": f"FINAL {champ} (refit 2020-23)", **{f"test_{k}": v for k, v in mf.items()}, "val_RMSE": np.nan, "val_R2": np.nan, "fit_seconds": np.nan})
        P["FINAL"] = pf
        joblib.dump({"classifier": clf_f, "amount_model": amt_f, "features": cols, "params": prm,
                     "usage": "rain_mm_expected = classifier.predict_proba(X)[:,1] * amount_model.predict(X)"},
                    f"{OUT}/models/rainfall_{scen}.joblib", compress=3)
        class _H:
            def fit(self, *a, **k): return self
            def predict(self, Z): return clf_f.predict_proba(Z)[:, 1] * amt_f.predict(Z)
        imp_model = _H()
    else:
        imp_model = rf
    # ---------- Permutation importance on 40k test rows
    ix = rng.choice(len(Xte), 40_000, replace=False)
    pi = permutation_importance(imp_model, Xte.iloc[ix], yte[ix], n_repeats=3, random_state=0, scoring=lambda est, Z, t_: -np.sqrt(np.mean((t_ - est.predict(Z)) ** 2)))
    importances[scen] = pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False)
    log("top features:", importances[scen].head(8).round(4).to_dict())
    preds_store[scen] = {"P": P, "champ": champ, "meta": mte.reset_index(drop=True), "y": yte, "cols": cols}

# ---------- Persist
pd.DataFrame(results).to_csv(f"{OUT}/results/model_comparison.csv", index=False)
pd.DataFrame(cls_results).to_csv(f"{OUT}/results/rain_occurrence_classification.csv", index=False)
pd.DataFrame(tuning_log).to_csv(f"{OUT}/results/hyperparameter_search.csv", index=False)
pd.concat(importances, axis=1).to_csv(f"{OUT}/results/permutation_importance.csv")
log("DONE")

## 6. Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(f"{OUT}/results/model_comparison.csv", index=False)
pd.DataFrame(cls_results).to_csv(f"{OUT}/results/rain_occurrence_classification.csv", index=False)
pd.DataFrame(tuning_log).to_csv(f"{OUT}/results/hyperparameter_search.csv", index=False)
pd.concat(importances, axis=1).to_csv(f"{OUT}/results/permutation_importance.csv")
res = results_df
display(results_df.round(4))
display(pd.DataFrame(cls_results).round(4))

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
F = f"{OUT}/figs"
store, imps = preds_store, importances
res = pd.read_csv(f"{OUT}/results/model_comparison.csv")

# 1 model comparison
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, sc in zip(axs, res.scenario.unique()):
    r = res[(res.scenario == sc) & ~res.model.str.startswith("FINAL")].set_index("model")
    cols = ["#adb5bd" if m.startswith("Baseline") else "#2a6f97" for m in r.index]
    ax.barh(r.index, r.test_RMSE, color=cols); ax.invert_yaxis(); ax.set_title(f"{sc}: test RMSE (2024, lower is better)")
    for i, (v, r2) in enumerate(zip(r.test_RMSE, r.test_R2)): ax.text(v, i, f" {v:.3f} (R²={r2:.2f})", va="center", fontsize=7)
    ax.set_xlim(0, r.test_RMSE.max() * 1.35)
plt.tight_layout(); plt.savefig(f"{F}/04_model_comparison.png"); plt.show()

# 2 daily totals scatter, 3 monthly profile
fig, axs = plt.subplots(2, 2, figsize=(11, 8.5))
for k, (sc, d) in enumerate(store.items()):
    p = d["P"]["FINAL"]; m = d["meta"]; y = d["y"]
    dd = pd.DataFrame({"k": m.lga + m.date.astype(str), "y": y, "p": p}).groupby("k").sum()
    ax = axs[0, k]; ax.scatter(dd.y, dd.p, s=3, alpha=.25, color="#2a6f97"); lim = max(dd.y.max(), dd.p.max())
    ax.plot([0, lim], [0, lim], "k--", lw=.8); ax.set_xlabel("observed daily total (mm)"); ax.set_ylabel("predicted"); ax.set_title(f"{sc}\n{d['champ']} – daily totals, test 2024")
    mm = pd.DataFrame({"month": m.month, "obs": y, "pred": p}).groupby("month").mean()
    ax = axs[1, k]; ax.plot(mm.index, mm.obs, "o-", label="observed", color="#bc4749"); ax.plot(mm.index, mm.pred, "s--", label="predicted", color="#2a6f97")
    ax.set_title("Monthly mean hourly rainfall, test 2024"); ax.set_xlabel("month"); ax.legend()
plt.tight_layout(); plt.savefig(f"{F}/05_daily_scatter_monthly_profile.png"); plt.show()

# 4 ROC / PR / calibration of rain-occurrence probabilities
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for sc, d, c in zip(store, store.values(), ["#2a6f97", "#bc4749"]):
    yb = (d["y"] > 0).astype(int); pr = d["P"]["_prob"]
    f, t, _ = roc_curve(yb, pr); axs[0].plot(f, t, color=c, label=f"{sc} (AUC {roc_auc_score(yb, pr):.3f})")
    a, b, _ = precision_recall_curve(yb, pr); axs[1].plot(b, a, color=c, label=f"{sc} (AP {average_precision_score(yb, pr):.3f})")
    bins = pd.qcut(pr, 15, duplicates="drop"); g = pd.DataFrame({"p": pr, "y": yb}).groupby(bins, observed=True).mean(); axs[2].plot(g.p, g.y, "o-", color=c, label=sc)
axs[0].plot([0, 1], [0, 1], "k--", lw=.7); axs[0].set_title("ROC – rain vs no rain"); axs[0].set_xlabel("FPR"); axs[0].set_ylabel("TPR"); axs[0].legend(fontsize=7)
axs[1].set_title("Precision–recall"); axs[1].set_xlabel("recall"); axs[1].set_ylabel("precision"); axs[1].legend(fontsize=7)
axs[2].plot([0, 1], [0, 1], "k--", lw=.7); axs[2].set_title("Calibration of P(rain)"); axs[2].set_xlabel("mean predicted"); axs[2].set_ylabel("observed frequency"); axs[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{F}/06_classification_curves.png"); plt.show()

# 5 permutation importance
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (sc, s) in zip(axs, imps.items()):
    s = s.head(15)[::-1]; ax.barh(s.index, s.values, color="#6a994e"); ax.set_title(f"{sc}\npermutation importance (ΔRMSE when shuffled)"); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.savefig(f"{F}/07_feature_importance.png"); plt.show()

# 6 hourly time series, wettest 10 days at Makurdi (scenario B), 7 per-LGA RMSE
d = store["B_with_rain_history"]; m = d["meta"]; sel = (m.lga == "Makurdi").values
ts = pd.DataFrame({"t": m.datetime[sel], "obs": d["y"][sel], "pred": d["P"]["FINAL"][sel]}).set_index("t")
w = ts.obs.resample("D").sum().rolling(5).sum().idxmax(); seg = ts[w - pd.Timedelta(days=4): w + pd.Timedelta(days=1)]
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
axs[0].bar(seg.index, seg.obs, width=.03, color="#bc4749", label="observed"); axs[0].plot(seg.index, seg.pred, color="#2a6f97", label="predicted"); axs[0].legend(); axs[0].set_title("Makurdi – wettest 5-day spell of 2024 (hourly, scenario B)")
r = pd.DataFrame({"lga": m.lga, "e": (d["y"] - d["P"]["FINAL"]) ** 2}).groupby("lga").e.mean().pow(.5).sort_values()
axs[1].barh(r.index, r.values, color="#2a6f97"); axs[1].tick_params(axis="y", labelsize=6); axs[1].set_title("Test RMSE by LGA (scenario B)")
plt.tight_layout(); plt.savefig(f"{F}/08_timeseries_and_lga_error.png"); plt.show()

## 7. Notes and caveats
- **Hourly rain is hard to predict:** rain *occurrence* is predicted well (ROC-AUC ≈ 0.95 / 0.97 for A / B) but *amounts* much less so (R² ≈ 0.3 weather-only, ≈ 0.5 with rain history).
- **Daily-total R² in scenario B is inflated** (even a 1-hour persistence baseline scores ≈ 0.99) – rely on the hourly metrics there.
- Validation RMSE is higher than test RMSE because 2023 was wetter than 2024; scores are year-dependent.
- Neighbouring LGAs are strongly correlated, so these results do not measure performance at an unseen location (try leave-LGA-out CV for that).
- Scenario B still uses concurrent weather at time *t*; a strict *forecast* would use only information up to *t−1* (natural extension).
- Load a saved model: `joblib.load(f"{OUT}/models/rainfall_<scenario>.joblib")` – hurdle models are dicts with `classifier`, `amount_model`, `features`; expected rain = `classifier.predict_proba(X)[:,1] * amount_model.predict(X)`.